# Automated Data Quality Validator

Detect missing values, duplicates, invalid ages, malformed emails, and invalid categories.


In [ ]:
import re
import pandas as pd


In [ ]:
df = pd.DataFrame([
    [1,"Amina","amina@example.com",22,"active"],
    [2,"Leo","bad-email",-5,"active"],
    [2,"Leo","bad-email",-5,"active"],
    [4,None,"maria@example.com",145,"unknown"],
    [5,"Noah","noah@example.com",31,"inactive"],
], columns=["id","name","email","age","status"])
df


In [ ]:
VALID_STATUS = {"active", "inactive"}
EMAIL_RE = re.compile(r"^[^@\s]+@[^@\s]+\.[^@\s]+$")

issues = []
for idx, row in df.iterrows():
    if row.isna().any():
        issues.append((idx, "missing_value", "One or more fields are missing"))
    if not (0 <= float(row["age"]) <= 120):
        issues.append((idx, "invalid_age", f"Age {row['age']} is outside 0–120"))
    if row["status"] not in VALID_STATUS:
        issues.append((idx, "invalid_status", f"Unexpected status: {row['status']}"))
    if not EMAIL_RE.match(str(row["email"])):
        issues.append((idx, "invalid_email", f"Malformed email: {row['email']}"))

for idx in df.index[df.duplicated(keep=False)]:
    issues.append((idx, "duplicate_row", "Row duplicates another record"))

report = pd.DataFrame(issues, columns=["row_index", "issue_type", "details"])
report


In [ ]:
report.to_csv("data_quality_report.csv", index=False)
